# Ocean Vision 101: Exploring FathomNet Plankton with FiftyOne

A beginner-friendly, self-contained notebook. If you've **never installed FiftyOne**, start here — the
first cell installs everything. You'll pull two real plankton image collections from
[FathomNet](https://www.fathomnet.org/), load them into [FiftyOne](https://docs.voxel51.com/), and
explore them with embeddings, similarity search, zero-shot classification, and an interactive confusion
matrix.

**What you need:** Python 3.9–3.12 and about 15 minutes. A laptop is fine — no GPU required (though Apple
Silicon / CUDA will speed up the embedding step). Everything runs locally.

**How to run:** execute the cells top to bottom. The install cell only needs to run once.

---


## 1. Install the packages

This installs FiftyOne (the dataset tool), the FathomNet Python client, PyTorch (for the CLIP model),
and UMAP (for the 2D embedding plot). Run once; re-running is harmless.

> If UMAP fails to install on your Python version, don't worry — the notebook automatically falls back
> to t-SNE, which needs nothing extra.


In [ ]:
import sys, subprocess

def pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)

pip_install(
    "fiftyone",       # dataset curation + app + Brain
    "fathomnet",      # FathomNet REST API client
    "torch",          # backend for the CLIP model
    "torchvision",
    "umap-learn",     # 2D embedding visualization (optional; tsne fallback exists)
    "requests",
    "pillow",
)
print("Install complete. If this is your first FiftyOne install, you're all set.")


### Quick sanity check
Confirm the key libraries import and report whether a GPU is available.


In [ ]:
import fiftyone as fo
import fiftyone.brain as fob
import torch

print("fiftyone version:", fo.__version__)
if torch.cuda.is_available():
    print("GPU: CUDA")
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    print("GPU: Apple Silicon (MPS)")
else:
    print("GPU: none (CPU is fine for 200 images)")


## 2. Configure the demo

These are the two FathomNet **collections** we'll load — small plankton image sets, ~100 images each.
Images download into a local folder. You can leave everything as-is.


In [ ]:
from pathlib import Path

# Two FathomNet plankton collections (whole-image classification crops).
COLLECTION_UUIDS = [
    "e601dec9-f99d-4596-b252-aa2f761d2a16",
    "6cdda2ef-fc17-472d-a56c-6d31504d4453",
]

# Local folder for downloaded images.
DATA_DIR = Path.home() / "fathomnet_plankton_demo" / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
print("Images will download to:", DATA_DIR)


## 3. Fetch the collections from FathomNet

FathomNet exposes images by collection through its REST API. We list each collection's images, then
fetch the full record (label + dimensions) for each. This makes ~200 small API calls, so give it a minute.

> **What's a "collection"?** In FathomNet, a collection is one upload/dataset. The long UUIDs above
> identify these two specifically. You can browse them in the FathomNet portal by pasting a UUID into
> the database viewer.


In [ ]:
from fathomnet.api import geoimages, images

records = []  # (collection_uuid, image record)
for cu in COLLECTION_UUIDS:
    listing = geoimages.find_by_image_set_upload_uuid(cu)
    print(f"Collection {cu[:8]}...: {len(listing)} images")
    for i, g in enumerate(listing):
        records.append((cu, images.find_by_uuid(g.uuid)))
        if (i + 1) % 50 == 0:
            print(f"  fetched metadata {i+1}/{len(listing)}")
print("Total images:", len(records))


## 4. Download the images locally

FiftyOne displays images from local files. This downloads each crop once (skipping any already present).


In [ ]:
import os, requests

def download(url, dest: Path):
    if dest.exists() and dest.stat().st_size > 0:
        return dest
    dest.parent.mkdir(parents=True, exist_ok=True)
    r = requests.get(url, timeout=60); r.raise_for_status()
    dest.write_bytes(r.content)
    return dest

paths = {}
for cu, rec in records:
    ext = os.path.splitext(rec.url)[1] or ".png"
    dest = DATA_DIR / cu / f"{rec.uuid}{ext}"
    try:
        paths[rec.uuid] = download(rec.url, dest)
    except Exception as e:
        print("skip", rec.uuid, e)
print(f"Downloaded {len(paths)} / {len(records)} images")


## 5. Build the FiftyOne dataset

Each image gets one **classification** label (the plankton concept assigned by expert annotators). We
combine both collections into a single dataset and keep `collection` as a field + tag so you can slice
by either one.

> **Why classification, not boxes?** These particular collections are whole-image crops with one label
> each — a classic image-classification setup, which is what the rest of the notebook assumes.


In [ ]:
COLL_NAMES = {
    COLLECTION_UUIDS[0]: "collection_A",
    COLLECTION_UUIDS[1]: "collection_B",
}

NAME = "fathomnet-plankton"
if NAME in fo.list_datasets():
    fo.delete_dataset(NAME)          # start fresh if re-running
ds = fo.Dataset(NAME, persistent=True)

samples = []
for cu, rec in records:
    if rec.uuid not in paths:
        continue
    boxes = getattr(rec, "boundingBoxes", None) or []
    concept = boxes[0].concept if boxes else "unlabeled"
    coll = COLL_NAMES.get(cu, cu)
    s = fo.Sample(filepath=str(paths[rec.uuid]), tags=[coll])
    s["collection"] = coll
    s["ground_truth"] = fo.Classification(label=concept)
    samples.append(s)

ds.add_samples(samples)
ds.compute_metadata()
print(ds)

print("\nConcepts in this dataset:")
for label, n in sorted(ds.count_values("ground_truth.label").items(), key=lambda kv: -kv[1]):
    print(f"  {n:4d}  {label}")


## 6. Open the FiftyOne App

This launches the interactive app right in the notebook. Try the **left sidebar**: expand
`ground_truth` to filter by plankton concept, or `collection` to compare the two uploads.

> The app also opens at http://localhost:5151 in your browser if you prefer a full window.


In [ ]:
session = fo.launch_app(ds)
session


## 7. Visualize the data with embeddings

We run each image through **CLIP** (a vision model) to get a numeric "fingerprint" (embedding), then
squash those into 2D so similar-looking plankton land near each other.

After it runs: in the app, click the **`+`** next to the *Samples* tab → **Embeddings** →
choose `clip_viz`, and set **color by** to `ground_truth.label`. Related taxa form neighborhoods; the
generic buckets (like `Diatom`) smear across them.

The embedding step is the slowest cell (a minute or two on CPU).


In [ ]:
import fiftyone.zoo as foz

clip = foz.load_zoo_model("clip-vit-base32-torch")
ds.compute_embeddings(clip, embeddings_field="clip_emb", batch_size=32)

# UMAP if available, otherwise fall back to t-SNE (no extra install needed).
try:
    fob.compute_visualization(ds, embeddings="clip_emb", method="umap", brain_key="clip_viz")
    print("Visualization ready (UMAP). Open the Embeddings panel and color by ground_truth.label")
except Exception as e:
    print("UMAP unavailable, using t-SNE instead:", e)
    fob.compute_visualization(ds, embeddings="clip_emb", method="tsne", brain_key="clip_viz")
    print("Visualization ready (t-SNE). Open the Embeddings panel and color by ground_truth.label")


## 8. Find visually similar images

Build a similarity index on the embeddings. Then, in the app, **select an image** (hover and click its
checkbox) and press the **similarity button** (looks like a stack of photos) to pull up its nearest
look-alikes — 'find me more plankton like this one'.

The cell below also shows the 10 nearest neighbors of the first image programmatically.


In [ ]:
fob.compute_similarity(ds, embeddings="clip_emb", brain_key="clip_sim")

first_id = ds.first().id
session.view = ds.sort_by_similarity(first_id, k=10, brain_key="clip_sim")
print("Showing 10 images most similar to a:", ds.first().ground_truth.label, "crop")


## 9. Classify images with zero-shot CLIP

Without training anything, we ask CLIP to guess each image's concept from the list of class names,
using the prompt template **"a microscope photo of {concept}"** (domain-specific prompts help). Guesses
go in a `predictions` field.

`store_logits=True` is important — it records a **confidence** score for each guess so you can sort and
filter by it in the app.


In [ ]:
classes = sorted(ds.distinct("ground_truth.label"))
print(f"{len(classes)} classes:", classes)

clip_zeroshot = foz.load_zoo_model(
    "clip-vit-base32-torch",
    text_prompt="a microscope photo of ",
    classes=classes,
)
ds.apply_model(clip_zeroshot, label_field="predictions", store_logits=True)

lo, hi = ds.bounds("predictions.confidence")
print(f"Predictions written. Confidence ranges {lo:.3f}–{hi:.3f}.")
print("(Low absolute confidence is normal across many fine-grained classes.)")


**Try it in the app:** make sure both `ground_truth` and `predictions` are toggled on in the sidebar
so each image shows the true label and CLIP's guess. Then filter `predictions.confidence` with the
min/max slider — high-confidence guesses are mostly the distinctive taxa; low-confidence ones pile up on
the vague catch-all buckets.


## 10. Score the predictions — the confusion matrix

Now compare CLIP's guesses to the expert labels. The **confusion matrix** is the payoff: it's clickable.
The diagonal is correct guesses; bright **off-diagonal** cells are confusions. Click one and the app
loads exactly those images so you can see *why* the model confused them.


In [ ]:
results = ds.evaluate_classifications(
    "predictions",
    gt_field="ground_truth",
    eval_key="clip_eval",
    classes=classes,
)
results.print_report()
print("\ncorrect vs incorrect:", ds.count_values("clip_eval"))

# Attach the interactive confusion matrix to the app.
plot = results.plot_confusion_matrix(classes=classes)
session.plots.attach(plot, name="confusion")
print("Confusion matrix attached — click an off-diagonal cell to inspect those images.")


> **If the matrix doesn't render**, install the widget backend once (`pip install anywidget`) and
> re-run this cell. To bring the matrix back in a later session without re-running the model:
> ```python
> results = ds.load_evaluation_results("clip_eval")
> plot = results.plot_confusion_matrix(classes=sorted(ds.distinct("ground_truth.label")))
> session.plots.attach(plot, name="confusion")
> ```


### See the biggest confusion directly
This finds CLIP's single most common mistake and loads those images — the same as clicking the brightest
off-diagonal cell.


In [ ]:
from fiftyone import ViewField as F

confusions = {}
for s in ds.match(F("clip_eval") == False):
    key = (s.ground_truth.label, s.predictions.label)
    confusions[key] = confusions.get(key, 0) + 1

if confusions:
    (true_lbl, pred_lbl), n = max(confusions.items(), key=lambda kv: kv[1])
    print(f"Biggest confusion: expert said '{true_lbl}', CLIP guessed '{pred_lbl}' ({n} images)")
    session.view = ds.match((F("ground_truth.label") == true_lbl) &
                            (F("predictions.label") == pred_lbl))
else:
    print("No misclassifications found.")


## 11. Find redundant / near-duplicate images

A **uniqueness** score (from the embeddings) flags images that look nearly the same as others — the kind
of redundancy you'd prune before training a model. Sorting ascending puts the most redundant first.

> In the app, try coloring the grid by `uniqueness` (palette icon → color by `uniqueness`). With only
> 200 images these are "most similar" rather than exact duplicates; on a full dataset of thousands this
> catches true duplicate frames.


In [ ]:
fob.compute_uniqueness(ds, embeddings="clip_emb")
session.view = ds.sort_by("uniqueness")   # most redundant first
print("uniqueness range:", ds.bounds("uniqueness"))


## 12. Save these views so you can reproduce them

Views set with `session.view = ...` disappear when the app closes. `save_view` stores them on the dataset
so they show up in the app's **view dropdown** (top-left) every time you open it. Run this once.


In [ ]:
# Clear old saved views so re-running stays clean.
for v in ds.list_saved_views():
    ds.delete_saved_view(v)

ds.save_view("low_confidence_predictions", ds.sort_by("predictions.confidence"),
             description="Least confident zero-shot guesses first")
ds.save_view("high_confidence_predictions", ds.sort_by("predictions.confidence", reverse=True),
             description="Most confident zero-shot guesses first")
ds.save_view("misclassifications",
             ds.match(F("clip_eval") == False).sort_by("predictions.confidence", reverse=True),
             description="Wrong guesses, most confident first")
ds.save_view("near_duplicates", ds.sort_by("uniqueness"),
             description="Most redundant images first")
for coll in ds.distinct("collection"):
    ds.save_view(f"collection__{coll}", ds.match(F("collection") == coll))

# The single biggest confusion cell, computed from the data.
confusions = {}
for s in ds.match(F("clip_eval") == False):
    key = (s.ground_truth.label, s.predictions.label)
    confusions[key] = confusions.get(key, 0) + 1
if confusions:
    (t, p), n = max(confusions.items(), key=lambda kv: kv[1])
    ds.save_view("top_confusion",
                 ds.match((F("ground_truth.label") == t) & (F("predictions.label") == p)),
                 description=f"Biggest confusion: {t} -> {p} ({n} images)")

print("Saved views (pick these from the app's view dropdown):")
for v in ds.list_saved_views():
    print("  -", v)


## 13. Wrap up

You've loaded a real ocean-imagery dataset, explored it visually, run a zero-shot model, and surfaced its
mistakes and redundancies — all locally.

**Keep exploring:**
- Re-open the app anytime with `fo.launch_app(fo.load_dataset("fathomnet-plankton"))` — the dataset,
  embeddings, and saved views all persist.
- Swap the zero-shot model for a fine-tuned classifier and re-run Section 10 to watch the confusion
  matrix tighten up.
- Point Section 2 at other FathomNet collections (many have bounding boxes, enabling object-detection
  workflows and map plots).

**Resources:**
- FathomNet: https://www.fathomnet.org/
- FathomNet Python client: https://github.com/fathomnet/fathomnet-py
- FiftyOne docs: https://docs.voxel51.com/
- FiftyOne Brain (embeddings, similarity, uniqueness): https://docs.voxel51.com/user_guide/brain.html

**Clean up (optional):** stop the app and free resources.


In [ ]:
# Optional cleanup
try:
    session.close()
except Exception:
    pass

# To delete the local dataset entirely, uncomment:
# fo.delete_dataset("fathomnet-plankton")
print("Done. Re-launch anytime with fo.load_dataset('fathomnet-plankton').")
